In [ ]:
# === CareConnect bootstrap — run me first ===
# Makes this notebook work from any folder and gives a clear error if the
# lab_helpers package is missing (e.g. not uploaded to the repo).
import os, sys

def _find_repo_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "lab_helpers")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "Could not find the 'lab_helpers/' folder from " + os.getcwd() + ".\n"
        "This means the helper package is not next to the notebooks.\n"
        "Fix: make sure lab_helpers/ and requirements.txt are in the same folder\n"
        "as these .ipynb files (see README > Setup). Then re-run this cell.")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

In [ ]:
# === Preflight: confirm every helper file is present BEFORE running the lab ===
import os
_required = [
    "requirements.txt",
    "lab_helpers/__init__.py",
    "lab_helpers/utils.py",
    "lab_helpers/careconnect_agents.py",
    "lab_helpers/deterministic_safety.py",
    "lab_helpers/runtime_entrypoint.py",
]
_missing = [f for f in _required if not os.path.isfile(f)]
if _missing:
    raise RuntimeError("Missing required files:\n  - " + "\n  - ".join(_missing) +
        "\n\nUpload the full lab_helpers/ folder + requirements.txt, then re-run.")
print("Preflight OK — all helper files present.")

# Lab 8: Cleanup — remove ONLY the `-sdk` resources

This deletes everything this SDK build created. It deliberately **does NOT touch**:
- the reused `careconnect-approved-docs` bucket (shared with your console build),
- anything without the `-sdk` suffix (your original build).

Run top to bottom. Each cell is defensive — missing resources are skipped.

### Step 1: Delete the AgentCore Runtime + ECR image

In [ ]:
import boto3
import lab_helpers.utils as u
acc = boto3.client("bedrock-agentcore-control", region_name=u.REGION)
try:
    arn = u.get_ssm_parameter(f"{u.SSM_PREFIX}/runtime_arn")
    rid = arn.split("/")[-1]
    acc.delete_agent_runtime(agentRuntimeId=rid)
    print("Deleted runtime", rid)
except Exception as e:
    print("Runtime delete skipped:", e)

### Step 2: Delete API Gateway + proxy Lambda

In [ ]:
apigw = boto3.client("apigateway", region_name=u.REGION)
lam = boto3.client("lambda", region_name=u.REGION)
for api in apigw.get_rest_apis().get("items", []):
    if api["name"] == u.name("careconnect-api"):
        apigw.delete_rest_api(restApiId=api["id"]); print("Deleted API", api["id"])
for fn in (u.PROXY_LAMBDA, u.MOCK_TOOLS_LAMBDA):
    try: lam.delete_function(FunctionName=fn); print("Deleted Lambda", fn)
    except Exception as e: print(fn, "skip:", e)

### Step 3: Delete Gateway + target

In [ ]:
try:
    gid = u.get_ssm_parameter(f"{u.SSM_PREFIX}/gateway_id")
    for t in acc.list_gateway_targets(gatewayIdentifier=gid).get("items", []):
        acc.delete_gateway_target(gatewayIdentifier=gid, targetId=t["targetId"])
    acc.delete_gateway(gatewayIdentifier=gid); print("Deleted gateway", gid)
except Exception as e:
    print("Gateway delete skipped:", e)

### Step 4: Delete Knowledge Base + confirm the S3 Vectors index is gone

In [ ]:
ba = boto3.client("bedrock-agent", region_name=u.REGION)
try:
    kb_id = u.get_ssm_parameter(f"{u.SSM_PREFIX}/kb_id")
    for ds in ba.list_data_sources(knowledgeBaseId=kb_id).get("dataSourceSummaries", []):
        ba.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds["dataSourceId"])
    ba.delete_knowledge_base(knowledgeBaseId=kb_id)
    print("Deleted KB", kb_id)
    print("NOW verify in S3 Vectors console that the vector index/bucket for the KB is gone.")
except Exception as e:
    print("KB delete skipped:", e)

### Step 5: Delete DynamoDB table + Step Functions state machine

In [ ]:
ddb = boto3.client("dynamodb", region_name=u.REGION)
sfn = boto3.client("stepfunctions", region_name=u.REGION)
try: ddb.delete_table(TableName=u.ESCALATION_TABLE); print("Deleted table")
except Exception as e: print("table skip:", e)
try:
    arn = u.get_ssm_parameter(f"{u.SSM_PREFIX}/state_machine_arn")
    sfn.delete_state_machine(stateMachineArn=arn); print("Deleted state machine")
except Exception as e: print("sfn skip:", e)

### Step 6: Delete the Guardrail

In [ ]:
bedrock = boto3.client("bedrock", region_name=u.REGION)
try:
    bedrock.delete_guardrail(guardrailIdentifier=u.get_ssm_parameter(f"{u.SSM_PREFIX}/guardrail_id"))
    print("Deleted guardrail")
except Exception as e: print("guardrail skip:", e)

### Step 7: Delete the `-sdk` IAM roles and SSM parameters

In [ ]:
iam = boto3.client("iam")
roles = [u.name(r) for r in ["CareConnectKBRole","CareConnectRuntimeRole",
    "CareConnectGatewayRole","CareConnectMockToolsRole","CareConnectProxyRole",
    "CareConnectSfnRole"]]
for r in roles:
    try:
        for p in iam.list_role_policies(RoleName=r)["PolicyNames"]:
            iam.delete_role_policy(RoleName=r, PolicyName=p)
        iam.delete_role(RoleName=r); print("Deleted role", r)
    except Exception as e: print(r, "skip:", e)

ssm = boto3.client("ssm")
for name in ["kb_id","guardrail_id","guardrail_version","gateway_id","gateway_url",
             "state_machine_arn","runtime_arn","api_url"]:
    u.delete_ssm_parameter(f"{u.SSM_PREFIX}/{name}")
print("Cleared SSM parameters.")

## Cleanup complete ✅

Removed only `-sdk` resources. The shared `careconnect-approved-docs` bucket and your
original console/CLI build are untouched.

**Not deleted on purpose:** the reused documents bucket, and (if you created one) any
customer-managed KMS key. Also remember: this SDK build ran in **SageMaker**, so when you
are fully done, stop/delete the SageMaker Studio space/app to stop notebook compute cost.